# Random Forest Comparison: With vs Without Temporal Features
This notebook compares Phoenix's original Random Forest setup (dropping `date`) against a version that adds numeric temporal features (`hour`, `day_of_week`, `month`) derived from `date`, while still dropping the raw timestamp.

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

SEED = 42
TEST_SIZE = 0.25
TARGET = "Appliances"
BASE_DEPTH = 5

In [4]:
# Load dataset
# Assumes energydata_complete.csv is in the same folder as this notebook OR provide a path below.
csv_path = "energydata_complete.csv"
data = pd.read_csv(csv_path)
data.head()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [5]:
def make_xy(df: pd.DataFrame, include_temporal: bool):
    df = df.copy()
    # Ensure datetime
    df["date"] = pd.to_datetime(df["date"])
    if include_temporal:
        df["hour"] = df["date"].dt.hour
        df["day_of_week"] = df["date"].dt.dayofweek
        df["month"] = df["date"].dt.month
    # Always drop raw timestamp (RF needs numeric)
    df = df.drop(columns=["date"])
    X = df.drop(columns=[TARGET]).values
    y = df[TARGET].values
    feature_names = df.drop(columns=[TARGET]).columns.tolist()
    return X, y, feature_names

In [ ]:
def tune_rf(X_train, y_train, X_test, y_test, base_depth=BASE_DEPTH):
    # 1) Baseline (single tree, shallow depth)
    baseline = RandomForestRegressor(
        n_estimators=1, max_depth=base_depth, random_state=SEED, n_jobs=-1
    )
    baseline.fit(X_train, y_train)
    pred = baseline.predict(X_test)
    baseline_mae = mean_absolute_error(y_test, pred)
    baseline_r2 = r2_score(y_test, pred)

    # 2) Tune n_estimators with fixed depth (range: 1..128)
    mae_list, r2_list, nums = [], [], []
    for i in range(1, 129):
        rf = RandomForestRegressor(
            n_estimators=i, max_depth=base_depth, random_state=SEED, n_jobs=-1
        )
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)
        mae_list.append(mean_absolute_error(y_test, y_pred))
        r2_list.append(r2_score(y_test, y_pred))
        nums.append(i)

    best_index = int(np.argmin(mae_list))
    best_trees = nums[best_index]

    # 3) Tune depth (Phoenix's range: 2..20) using best_trees
    depths = list(range(2, 21))
    depth_mae, depth_r2 = [], []
    for d in depths:
        rf_depth = RandomForestRegressor(
            n_estimators=best_trees, max_depth=d, random_state=SEED, n_jobs=-1
        )
        rf_depth.fit(X_train, y_train)
        y_pred = rf_depth.predict(X_test)
        depth_mae.append(mean_absolute_error(y_test, y_pred))
        depth_r2.append(r2_score(y_test, y_pred))

    best_depth_idx = int(np.argmin(depth_mae))
    best_depth = depths[best_depth_idx]

    # 4) Final model
    final = RandomForestRegressor(
        n_estimators=best_trees, max_depth=best_depth, random_state=SEED, n_jobs=-1
    )
    final.fit(X_train, y_train)
    final_pred = final.predict(X_test)
    final_mae = mean_absolute_error(y_test, final_pred)
    final_r2 = r2_score(y_test, final_pred)

    return {
        "baseline_mae": baseline_mae,
        "baseline_r2": baseline_r2,
        "best_trees": best_trees,
        "best_depth": best_depth,
        "final_mae": final_mae,
        "final_r2": final_r2,
        "final_model": final,
    }

In [7]:
def run_experiment(include_temporal: bool):
    X, y, feature_names = make_xy(data, include_temporal=include_temporal)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED
    )

    res = tune_rf(X_train, y_train, X_test, y_test)
    res["include_temporal"] = include_temporal
    res["feature_names"] = feature_names
    res["n_features"] = len(feature_names)
    return res

res_no_time = run_experiment(include_temporal=False)
res_with_time = run_experiment(include_temporal=True)

In [8]:
# Side-by-side comparison (main output for report)
comparison = pd.DataFrame([
    {
        "Model": "RF (original: no temporal)",
        "Temporal features": "No",
        "Num features": res_no_time["n_features"],
        "Baseline MAE": res_no_time["baseline_mae"],
        "Baseline R²": res_no_time["baseline_r2"],
        "Final MAE": res_no_time["final_mae"],
        "Final R²": res_no_time["final_r2"],
        "Best n_estimators": res_no_time["best_trees"],
        "Best max_depth": res_no_time["best_depth"],
    },
    {
        "Model": "RF (+ temporal: hour/day/month)",
        "Temporal features": "Yes",
        "Num features": res_with_time["n_features"],
        "Baseline MAE": res_with_time["baseline_mae"],
        "Baseline R²": res_with_time["baseline_r2"],
        "Final MAE": res_with_time["final_mae"],
        "Final R²": res_with_time["final_r2"],
        "Best n_estimators": res_with_time["best_trees"],
        "Best max_depth": res_with_time["best_depth"],
    }
])

comparison

,Model,Temporal features,Num features,Baseline MAE,Baseline R²,Final MAE,Final R²,Best n_estimators,Best max_depth
0,RF (original: no temporal),No,27,52.909162,0.077995,34.167162,0.501838,25,20
1,RF (+ temporal: hour/day/month),Yes,30,49.560019,0.161260,33.549486,0.517277,58,20


In [ ]:
# check whether temporal variables appear among the most important features
def top_feature_importance(res, top_n=10):
    model = res["final_model"]
    names = res["feature_names"]
    importances = pd.Series(model.feature_importances_, index=names).sort_values(ascending=False)
    return importances.head(top_n)

print("Top 10 features (no temporal):")
display(top_feature_importance(res_no_time, 10))

print("\nTop 10 features (with temporal):")
display(top_feature_importance(res_with_time, 10))

Top 10 features (no temporal):


RH_1           0.064780
RH_out         0.061092
Press_mm_hg    0.055246
RH_8           0.052449
RH_2           0.048987
RH_5           0.046284
lights         0.046043
RH_3           0.045368
T3             0.043952
RH_6           0.043398
dtype: float64


Top 10 features (with temporal):


hour           0.154832
T3             0.060486
Press_mm_hg    0.043329
RH_3           0.042097
RH_5           0.042058
T8             0.038538
RH_2           0.034452
RH_1           0.033663
RH_7           0.032523
Tdewpoint      0.031442
dtype: float64